# BTC price (91 Days) Data Extraction, Load to BigQuery

In [ ]:
# import libraries
import requests
import pandas as pd
import os
from dotenv import load_dotenv
import pandas_gbq

## Data Extraction from Coingecko

In [19]:
# load API key from .env file
load_dotenv()   
API_KEY = os.environ.get('GECKO_API_KEY')
if not API_KEY:
    raise ValueError("Error: GECKO_API_KEY environment variable not set.")

# set parameters for API request
COIN_ID = "bitcoin"
VS_CURRENCY = "usd"
DAYS = 91  # last 90 days   
url = f"https://api.coingecko.com/api/v3/coins/{COIN_ID}/market_chart"

params = {
    "vs_currency": VS_CURRENCY,
    "days": DAYS,
    "interval": "daily"  # daily prices
}

headers = {"x-cg-api-key": API_KEY}

# make API request
response = requests.get(url, params=params, headers=headers)    
data = response.json()
print(data.keys()) 

# print first 5 price entries
data['prices'][:5]


dict_keys(['prices', 'market_caps', 'total_volumes'])


[[1755648000000, 112778.34483555844],
 [1755734400000, 114252.39755195397],
 [1755820800000, 112414.39987336512],
 [1755907200000, 116834.24948202295],
 [1755993600000, 115359.98346714744]]

In [20]:
# convert price data to DataFrame and set column names
prices = pd.DataFrame(data['prices'], columns=['timestamp', 'price'])
prices.head()

,timestamp,price
0,1755648000000,112778.344836
1,1755734400000,114252.397552
2,1755820800000,112414.399873
3,1755907200000,116834.249482
4,1755993600000,115359.983467


In [21]:
# convert timestamp(ms) to datetime 
prices['date'] = pd.to_datetime(prices['timestamp'], unit='ms') 
prices.head()

,timestamp,price,date
0,1755648000000,112778.344836,2025-08-20
1,1755734400000,114252.397552,2025-08-21
2,1755820800000,112414.399873,2025-08-22
3,1755907200000,116834.249482,2025-08-23
4,1755993600000,115359.983467,2025-08-24


In [22]:
# select relevant columns
prices_df = prices[['date', 'price']]

# keep the first 90 rows, which are all clean, closed prices.
prices_df = prices_df.iloc[:-1]

# final columns
prices_df = prices_df[['date', 'price']]

prices_df.tail()

,date,price
86,2025-11-14,99730.453396
87,2025-11-15,94456.393682
88,2025-11-16,95508.310199
89,2025-11-17,94411.329472
90,2025-11-18,92036.725505


## Loading data into Warehouse (BigQuery)

In [23]:
# setting up parameters to load data to GBQ
GCP_PROJECT_ID = "silken-apex-477018-d4"
destination_table = "btc.raw_coingecko_bitcoin"
    
print(f"Attempting to load data into BigQuery table: {destination_table}")

# Loading Data
pandas_gbq.to_gbq(
    prices_df,
    destination_table=destination_table,
    project_id=GCP_PROJECT_ID,
    if_exists='replace'
)

print("✅ Success! Data loaded into BigQuery.")

Attempting to load data into BigQuery table: btc.raw_coingecko_bitcoin


100%|██████████| 1/1 [00:00<?, ?it/s]

✅ Success! Data loaded into BigQuery.
